# Reference

In [1]:
from transformers import pipeline
model = pipeline( task="fill-mask", model="distilbert/distilbert-base-uncased" )
model( "Paris is the [MASK] of France" )

2025-09-27 15:15:22.191723: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758986122.436312      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758986122.506516      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


[{'score': 0.9741511940956116,
  'token': 3007,
  'token_str': 'capital',
  'sequence': 'paris is the capital of france'},
 {'score': 0.008014173246920109,
  'token': 14508,
  'token_str': 'birthplace',
  'sequence': 'paris is the birthplace of france'},
 {'score': 0.0014765257947146893,
  'token': 2803,
  'token_str': 'centre',
  'sequence': 'paris is the centre of france'},
 {'score': 0.0012379804393276572,
  'token': 18236,
  'token_str': 'metropolis',
  'sequence': 'paris is the metropolis of france'},
 {'score': 0.0009564656647853553,
  'token': 2835,
  'token_str': 'seat',
  'sequence': 'paris is the seat of france'}]

# Manual implementing

In [2]:
%%capture
from transformers import DistilBertTokenizer, DistilBertForMaskedLM

model = DistilBertForMaskedLM.from_pretrained("distilbert/distilbert-base-uncased")
tokenizer = DistilBertTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

In [3]:
text = "Paris is the [MASK] of France."
imp = tokenizer( text, padding=True, return_tensors="pt" )

In [4]:
imp

{'input_ids': tensor([[ 101, 3000, 2003, 1996,  103, 1997, 2605, 1012,  102]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [5]:
out = model( **imp )
out

MaskedLMOutput(loss=None, logits=tensor([[[ -5.3132,  -5.3143,  -5.3445,  ...,  -4.7040,  -4.5371,  -2.7950],
         [ -8.5938,  -8.9039,  -8.7784,  ...,  -7.9367,  -7.5802,  -5.0010],
         [ -8.3131,  -8.5002,  -8.1927,  ...,  -7.2223,  -4.7377,  -8.3184],
         ...,
         [ -9.3771,  -9.7591,  -9.5005,  ...,  -7.5939,  -7.1719,  -6.8949],
         [ -9.7038,  -9.6029,  -9.6775,  ...,  -7.9809,  -8.2816,  -4.8104],
         [-11.7981, -11.8289, -11.8431,  ..., -11.3846, -10.6421,  -7.2887]]],
       grad_fn=<ViewBackward0>), hidden_states=None, attentions=None)

In [6]:
print( out.logits.shape )
print( out.logits )

torch.Size([1, 9, 30522])
tensor([[[ -5.3132,  -5.3143,  -5.3445,  ...,  -4.7040,  -4.5371,  -2.7950],
         [ -8.5938,  -8.9039,  -8.7784,  ...,  -7.9367,  -7.5802,  -5.0010],
         [ -8.3131,  -8.5002,  -8.1927,  ...,  -7.2223,  -4.7377,  -8.3184],
         ...,
         [ -9.3771,  -9.7591,  -9.5005,  ...,  -7.5939,  -7.1719,  -6.8949],
         [ -9.7038,  -9.6029,  -9.6775,  ...,  -7.9809,  -8.2816,  -4.8104],
         [-11.7981, -11.8289, -11.8431,  ..., -11.3846, -10.6421,  -7.2887]]],
       grad_fn=<ViewBackward0>)


In [7]:
import torch

to_decode = []

out = out.logits.squeeze().softmax(dim=-1)
for i in range(out.shape[0]):
    print( out[i].max().item() )
    print( out[i].argmax().item() )
    to_decode.append( out[i].argmax().item() )
    print()

0.02209414727985859
1012

0.6569573283195496
3000

0.996338963508606
2003

0.9918094277381897
1996

0.9815466403961182
3007

0.9895901083946228
1997

0.9978752136230469
2605

0.9999927282333374
1012

0.6306298971176147
1012



In [8]:
tokenizer.decode( to_decode )

'. paris is the capital of france..'